In [3]:
# Setup -- SQL right inside this notebook via sqlite3
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect(':memory:')

def q(sql):
    return pd.read_sql_query(sql, conn)

def run(sql):
    conn.executescript(sql)
    conn.commit()

# Path to the 4 CSVs (downloaded from the course site)
DATA_DIR = os.path.expanduser('./data')

print('SQL engine ready.  sqlite3 version:', sqlite3.sqlite_version)
print('Data dir            :', DATA_DIR)
print('Files present       :', sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else 'NOT FOUND')

SQL engine ready.  sqlite3 version: 3.37.2
Data dir            : ./data
Files present       : ['.ipynb_checkpoints', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [4]:
customers = pd.read_csv('./data/olist_customers_dataset.csv')
orders = pd.read_csv('./data/olist_orders_dataset.csv')
order_items = pd.read_csv('./data/olist_order_items_dataset.csv')
payments = pd.read_csv('./data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('./data/olist_order_reviews_dataset.csv')
products = pd.read_csv('./data/olist_products_dataset.csv')
sellers = pd.read_csv('./data/olist_sellers_dataset.csv')
geolocation = pd.read_csv('./data/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('./data/product_category_name_translation.csv')

In [5]:
customers.to_sql('customers', conn, index=False, if_exists='replace')
orders.to_sql('orders', conn, index=False, if_exists='replace')
order_items.to_sql('order_items', conn, index=False, if_exists='replace')
payments.to_sql('payments', conn, index=False, if_exists='replace')
reviews.to_sql('reviews', conn, index=False, if_exists='replace')
products.to_sql('products', conn, index=False, if_exists='replace')
sellers.to_sql('sellers', conn, index=False, if_exists='replace')
geolocation.to_sql('geolocation', conn, index=False, if_exists='replace')
category_translation.to_sql('category_translation', conn, index=False, if_exists='replace')

71

In [8]:
tables = q('''
SELECT name
FROM sqlite_master
WHERE type = 'table';
''')


for table in tables['name']:
    cols = q(f'PRAGMA table_info({table});')['name'].tolist()
    print(table, ':', cols)

customers : ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
orders : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
order_items : ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments : ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews : ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
products : ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
sellers : ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
geolocation : ['geoloc

In [9]:
# Query 1: Count and Percentage of Orders Purchased in Jan 2018 with 5 Review Score
# Write and execute a SQL query to count the number of orders purchased in January 2018 that have a review score of 5 and calculate the percentage of such orders.


q('''
SELECT COUNT(*) AS january_2018_orders
FROM orders
JOIN reviews
ON orders.order_id = reviews.order_id
WHERE order_purchase_timestamp >= '2018-01-01'
  AND order_purchase_timestamp < '2018-02-01'
  AND review_score==5

''')

,january_2018_orders
0,4097


In [11]:
# Query 2: Customer Purchase Trend Year-on-Year
# Write and execute a SQL query to analyze the customer purchase trend year-on-year.

q('''
SELECT strftime('%Y', order_purchase_timestamp) AS purchase_year,
       COUNT(*) AS total_orders
from orders
group by purchase_year
order by purchase_year

''')

,purchase_year,total_orders
0,2016,329
1,2017,45101
2,2018,54011


In [19]:
# Query 3: Average Order Values of Customers
# Write and execute a SQL query to calculate the average order values of customers.

q('''
SELECT customer_id , ROUND(AVG(payment_value), 2) AS avg_order_value
from orders
join payments
on orders.order_id = payments.order_id
group by customer_id
ORDER BY avg_order_value DESC;
''')



,customer_id,avg_order_value
0,1617b1357756262bfa56ab541c47bc16,13664.08
1,ec5b2ba62e574342386871631fafd3fc,7274.88
2,c6e2731c5b391845f6800c97401a43a9,6929.31
3,f48d464a0baaea338cb25f816991ab1f,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,6726.66
...,...,...
99435,b246eeed30b362c09d867b9e598bee51,1.86
99436,fd123d346a17cdf5e37a2a85501069bf,1.74
99437,a73c1f73f5772cf801434bf984b0b1a7,0.00
99438,3532ba38a3fd242259a514ac2b6ae6b6,0.00


In [35]:
# Query 4: Top 5 Cities with Highest Revenue from 2016 to 2018
# Write and execute a SQL query to find the top 5 cities with the highest revenue from 2016 to 2018.

q("""
SELECT
    s.seller_city,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN sellers s
    ON oi.seller_id = s.seller_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.order_purchase_timestamp >= '2016-01-01'
  AND o.order_purchase_timestamp < '2019-01-01'
GROUP BY s.seller_city
ORDER BY revenue DESC
LIMIT 5;
""")

,seller_city,revenue
0,sao paulo,2702878.14
1,ibitinga,624592.94
2,curitiba,470759.82
3,rio de janeiro,358413.59
4,guarulhos,329494.38


In [36]:
# Query 5: State Wise Revenue Table Between 2016 to 2018
# Write and execute a SQL query to create a state-wise revenue table between 2016 to 2018.


q("""
SELECT
    s.seller_state,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN sellers s
    ON oi.seller_id = s.seller_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.order_purchase_timestamp >= '2016-01-01'
  AND o.order_purchase_timestamp < '2019-01-01'
GROUP BY s.seller_state
ORDER BY revenue DESC;
""")


,seller_state,revenue
0,SP,8753396.21
1,PR,1261887.21
2,MG,1011564.74
3,RJ,843984.22
4,SC,632426.07
5,RS,378559.54
6,BA,285561.56
7,DF,97749.48
8,PE,91493.85
9,GO,66399.21


In [41]:
# Query 6: Top Successful Sellers in Terms of Goods Sold, Revenue, and Customer Count
# Write and execute a SQL query to identify the top successful sellers in terms of the number of goods sold, total revenue, customer count, and sellers with the highest 5-star ratings.

q("""
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,

    COUNT(oi.order_item_id) AS goods_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    COUNT(DISTINCT o.customer_id) AS customer_count,
    SUM(CASE WHEN r.review_score = 5 THEN 1 ELSE 0 END) AS five_star_reviews

FROM order_items oi
JOIN sellers s
    ON oi.seller_id = s.seller_id
JOIN orders o
    ON oi.order_id = o.order_id
LEFT JOIN reviews r
    ON oi.order_id = r.order_id

GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state

ORDER BY
    goods_sold DESC,
    total_revenue DESC,
    customer_count DESC,
    five_star_reviews DESC

LIMIT 10;
""")

,seller_id,seller_city,seller_state,goods_sold,total_revenue,customer_count,five_star_reviews
0,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,2039,123585.82,1854,1024
1,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,2009,202999.12,1806,947
2,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1940,107431.41,1404,1096
3,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,1819,106555.98,1706,1053
4,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1574,162723.37,1314,893
5,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1501,135241.70,1287,804
6,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1443,140513.14,915,729
7,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1375,189417.67,982,437
8,ea8482cd71df3c1969d7b9473ff13abc,sao paulo,SP,1204,37205.51,1146,620
9,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1175,142325.49,1160,717


In [40]:
# Query 7: Delivery Success Rate Across States
# Write and execute a SQL query to calculate the delivery success rate across different states.

q("""
SELECT
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    ROUND(
        SUM(CASE WHEN o.order_status = 'delivered' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS delivery_success_rate
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY delivery_success_rate DESC;
""")

,customer_state,total_orders,delivered_orders,delivery_success_rate
0,AC,81,80,98.77
1,AP,68,67,98.53
2,ES,2033,1995,98.13
3,MS,715,701,98.04
4,AM,148,145,97.97
5,TO,280,274,97.86
6,RS,5466,5345,97.79
7,RN,485,474,97.73
8,MT,907,886,97.68
9,PR,5045,4923,97.58


In [49]:
# Query 8: Preferred Form of Payment for Different Categories
# Write and execute a SQL query to find the preferred form of payment for different product categories.


q("""
WITH payment_counts AS (
    SELECT
        ct.product_category_name_english AS product_category,
        pmt.payment_type,
        COUNT(*) AS payment_count
    FROM payments pmt
    JOIN order_items oi
        ON pmt.order_id = oi.order_id
    JOIN products p
        ON oi.product_id = p.product_id
    JOIN category_translation ct
        ON p.product_category_name = ct.product_category_name
    GROUP BY
        ct.product_category_name_english,
        pmt.payment_type
),

ranked AS (
    SELECT
        product_category,
        payment_type,
        payment_count,
        ROW_NUMBER() OVER (
            PARTITION BY product_category
            ORDER BY payment_count DESC
        ) AS rn
    FROM payment_counts
)

SELECT
    product_category,
    payment_type AS preferred_payment_type,
    payment_count
FROM ranked
WHERE rn = 1
ORDER BY payment_count DESC;
""")

,product_category,preferred_payment_type,payment_count
0,bed_bath_table,credit_card,8959
1,health_beauty,credit_card,7566
2,sports_leisure,credit_card,6635
3,furniture_decor,credit_card,6379
4,computers_accessories,credit_card,5436
...,...,...,...
66,arts_and_craftmanship,credit_card,14
67,la_cuisine,credit_card,13
68,cds_dvds_musicals,credit_card,9
69,fashion_childrens_clothes,credit_card,5


In [50]:
# Query 9: Distance Between Cities
# Write and execute a SQL query to calculate the distance between cities.



q("""
WITH city_locations AS (
    SELECT
        geolocation_city,
        geolocation_state,
        AVG(geolocation_lat) AS lat,
        AVG(geolocation_lng) AS lng
    FROM geolocation
    GROUP BY geolocation_city, geolocation_state
),

city_pair AS (
    SELECT
        c1.geolocation_city AS city_1,
        c1.geolocation_state AS state_1,
        c1.lat AS lat_1,
        c1.lng AS lng_1,
        c2.geolocation_city AS city_2,
        c2.geolocation_state AS state_2,
        c2.lat AS lat_2,
        c2.lng AS lng_2
    FROM city_locations c1
    JOIN city_locations c2
    WHERE c1.geolocation_city = 'sao paulo'
      AND c1.geolocation_state = 'SP'
      AND c2.geolocation_city = 'rio de janeiro'
      AND c2.geolocation_state = 'RJ'
)

SELECT
    city_1,
    state_1,
    city_2,
    state_2,
    ROUND(
        6371 * 2 * ASIN(
            SQRT(
                POWER(SIN(((lat_2 - lat_1) * 3.141592653589793 / 180) / 2), 2)
                +
                COS(lat_1 * 3.141592653589793 / 180)
                * COS(lat_2 * 3.141592653589793 / 180)
                * POWER(SIN(((lng_2 - lng_1) * 3.141592653589793 / 180) / 2), 2)
            )
        ),
        2
    ) AS distance_km
FROM city_pair;
""")

,city_1,state_1,city_2,state_2,distance_km
0,sao paulo,SP,rio de janeiro,RJ,346.99
